# Grammar Scoring Engine — Speech Preprocessing Pipeline

**Goal:** prepare 45–60 s spoken audio clips so a model can predict a **grammar MOS score (0–5)**.

**Why preprocessing matters here:** grammar is judged from *what* the speaker says, so the most important output of this stage is a **faithful transcript**. Every audio step below is chosen to help speech recognition (Whisper) hear the speaker clearly **without "correcting" their mistakes**. We also extract **fluency features** (pauses, speech rate) because human raters are influenced by them.

| Step | Method | Why |
|---|---|---|
| 1 | Data discovery & integrity check | Catch missing/corrupt files before training |
| 2 | Exploratory analysis | Know label distribution and clip durations |
| 3 | Resample to 16 kHz mono | Required input format for Whisper / wav2vec2 |
| 4 | DC-offset removal | Removes microphone bias that distorts energy features |
| 5 | Leading/trailing silence trimming | Removes dead air without touching in-speech pauses |
| 6 | Optional noise reduction | Cleans noisy recordings (applied only when SNR is low) |
| 7 | Loudness normalisation | Makes quiet and loud speakers comparable |
| 8 | Quality metrics (SNR, clipping, speech ratio) | Flags bad recordings; useful as features |
| 9 | Voice-activity / pause analysis | Fluency features: pauses, speech ratio |
| 10 | Verbatim transcription (faster-whisper) | Turns speech into text **keeping errors and fillers** |
| 11 | Transcript cleaning & text features | Word count, speech rate, fillers, repetitions |
| 12 | Stratified CV folds | Fair evaluation on only 769 samples |

**Kaggle settings:** turn on **GPU (T4 or P100)** and **Internet** in the notebook settings (right panel) before running.

## 0. Install & import libraries

In [ ]:
!pip install -q faster-whisper noisereduce pyloudnorm

In [ ]:
import os, glob, re, json, warnings
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

TARGET_SR = 16000                      # Whisper / wav2vec2 expect 16 kHz
WORK_DIR = "/kaggle/working"
CLEAN_DIR = os.path.join(WORK_DIR, "clean_audio")
os.makedirs(CLEAN_DIR, exist_ok=True)

## 1. Find the dataset automatically

Kaggle mounts competition data under `/kaggle/input/`. Folder names differ between competitions, so this cell **searches for the CSV files and audio folders** instead of hard-coding paths. Check the printed output and fix the variables in the next cell if anything is detected wrongly.

In [ ]:
INPUT_ROOT = "/kaggle/input"
AUDIO_EXT = (".wav", ".mp3", ".flac", ".m4a", ".ogg")

csv_files = sorted(glob.glob(f"{INPUT_ROOT}/**/*.csv", recursive=True))
audio_files = [p for p in glob.glob(f"{INPUT_ROOT}/**/*", recursive=True) if p.lower().endswith(AUDIO_EXT)]
audio_dirs = sorted({os.path.dirname(p) for p in audio_files})

print("CSV files found:")
for c in csv_files: print("  ", c)
print("\nAudio folders found (count):")
for d in audio_dirs:
    print(f"   {d}  ->  {sum(1 for p in audio_files if os.path.dirname(p)==d)} files")

In [ ]:
def pick(paths, keyword):
    hits = [p for p in paths if keyword in os.path.basename(p).lower() or keyword in p.lower().split("/")[-2:][0]]
    return hits[0] if hits else None

TRAIN_CSV = pick(csv_files, "train")
TEST_CSV  = pick(csv_files, "test")
TRAIN_AUDIO_DIR = next((d for d in audio_dirs if "train" in d.lower()), audio_dirs[0] if audio_dirs else None)
TEST_AUDIO_DIR  = next((d for d in audio_dirs if "test"  in d.lower()), None)

# ---- Override here if auto-detection picked the wrong paths ----
# TRAIN_CSV = "/kaggle/input/<competition>/train.csv"
# TEST_CSV = "/kaggle/input/<competition>/test.csv"
# TRAIN_AUDIO_DIR = "/kaggle/input/<competition>/audios/train"
# TEST_AUDIO_DIR = "/kaggle/input/<competition>/audios/test"

print("TRAIN_CSV       :", TRAIN_CSV)
print("TEST_CSV        :", TEST_CSV)
print("TRAIN_AUDIO_DIR :", TRAIN_AUDIO_DIR)
print("TEST_AUDIO_DIR  :", TEST_AUDIO_DIR)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV) if TEST_CSV else pd.DataFrame()

# Detect the filename and label columns
FILE_COL = next(c for c in train_df.columns if any(k in c.lower() for k in ["file", "name", "audio", "id"]))
LABEL_COL = next((c for c in train_df.columns if any(k in c.lower() for k in ["label", "score", "grammar", "mos"])), None)
print("Filename column:", FILE_COL, "| Label column:", LABEL_COL)
display(train_df.head())

## 2. Integrity check

Before any processing: does every row in the CSV have an audio file, and can every file be opened? A single corrupt file can crash a 2-hour transcription run halfway.

In [ ]:
def resolve_path(name, folder):
    # Match a CSV filename to an actual audio file (handles missing extensions).
    name = str(name)
    direct = os.path.join(folder, name)
    if os.path.exists(direct):
        return direct
    for ext in AUDIO_EXT:
        if os.path.exists(direct + ext):
            return direct + ext
    return None

train_df["path"] = train_df[FILE_COL].apply(lambda n: resolve_path(n, TRAIN_AUDIO_DIR))
if len(test_df):
    TEST_FILE_COL = FILE_COL if FILE_COL in test_df.columns else test_df.columns[0]
    test_df["path"] = test_df[TEST_FILE_COL].apply(lambda n: resolve_path(n, TEST_AUDIO_DIR))

def audio_info(path):
    try:
        info = sf.info(path)
        return pd.Series({"orig_sr": info.samplerate, "channels": info.channels,
                          "duration": info.duration, "readable": True})
    except Exception:
        try:  # fall back to librosa for formats soundfile cannot read (e.g. mp3)
            y, sr = librosa.load(path, sr=None, mono=False)
            ch = 1 if y.ndim == 1 else y.shape[0]
            return pd.Series({"orig_sr": sr, "channels": ch,
                              "duration": y.shape[-1] / sr, "readable": True})
        except Exception:
            return pd.Series({"orig_sr": np.nan, "channels": np.nan, "duration": np.nan, "readable": False})

for name, df in [("train", train_df), ("test", test_df)]:
    if not len(df): continue
    missing = df["path"].isna().sum()
    info = df["path"].dropna().apply(audio_info)
    df[info.columns] = info
    print(f"{name}: {len(df)} rows | missing files: {missing} | unreadable: {(df['readable'] == False).sum()}")

## 3. Exploratory analysis

- **Label distribution** — tells us if scores are imbalanced (e.g. few 1s and 5s), which affects how we build CV folds.
- **Durations & sample rates** — confirms the 45–60 s range and shows whether resampling is needed.
- **Duplicates / outliers** — very short clips are often recording failures.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
if LABEL_COL:
    train_df[LABEL_COL].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#4C72B0")
    axes[0].set_title("Grammar score distribution (train)"); axes[0].set_xlabel("score")
train_df["duration"].plot(kind="hist", bins=30, ax=axes[1], color="#55A868", alpha=.8, label="train")
if len(test_df): test_df["duration"].plot(kind="hist", bins=30, ax=axes[1], color="#C44E52", alpha=.6, label="test")
axes[1].set_title("Clip duration (s)"); axes[1].legend()
train_df["orig_sr"].value_counts().plot(kind="bar", ax=axes[2], color="#8172B2")
axes[2].set_title("Original sample rates")
plt.tight_layout(); plt.show()

if LABEL_COL: print(train_df[LABEL_COL].describe())
print("\nChannels:", train_df["channels"].value_counts().to_dict())
print("Clips shorter than 20 s:", (train_df["duration"] < 20).sum())

In [ ]:
# Listen to a low, middle and high scoring sample (helps you understand the rubric)
from IPython.display import Audio, display
if LABEL_COL:
    for q in [0.05, 0.5, 0.95]:
        row = train_df.iloc[(train_df[LABEL_COL] - train_df[LABEL_COL].quantile(q)).abs().argsort().iloc[0]]
        print(f"score = {row[LABEL_COL]}  |  {os.path.basename(row['path'])}")
        display(Audio(row["path"]))

## 4. Audio preprocessing functions

Each function does one job so you can explain (and ablate) each step in your report.

**Important design choice:** we **do not** remove pauses *inside* speech, change the speed, or apply heavy filtering. Hesitations and pauses carry information about the speaker's language ability, and aggressive cleaning can make Whisper hallucinate or "fix" grammar.

In [ ]:
import noisereduce as nr
import pyloudnorm as pyln

def load_audio(path, sr=TARGET_SR):
    # Step 3: load as mono and resample to 16 kHz (high-quality resampler).
    y, _ = librosa.load(path, sr=sr, mono=True, res_type="soxr_hq")
    return y.astype(np.float32)

def remove_dc_offset(y):
    # Step 4: subtract the mean so the waveform is centred on zero.
    return y - np.mean(y)

def trim_edges(y, top_db=35, pad_s=0.25, sr=TARGET_SR):
    # Step 5: trim silence only at the start/end, keeping a small padding.
    _, (start, end) = librosa.effects.trim(y, top_db=top_db)
    pad = int(pad_s * sr)
    return y[max(0, start - pad): min(len(y), end + pad)]

def estimate_snr(y, sr=TARGET_SR, top_db=30):
    # Rough SNR: energy in speech regions vs energy in non-speech regions (dB).
    intervals = librosa.effects.split(y, top_db=top_db)
    if len(intervals) == 0:
        return 0.0
    mask = np.zeros(len(y), dtype=bool)
    for s, e in intervals: mask[s:e] = True
    speech, noise = y[mask], y[~mask]
    if len(noise) < sr * 0.2:          # almost no silence -> cannot estimate, assume clean
        return 40.0
    p_s, p_n = np.mean(speech**2) + 1e-12, np.mean(noise**2) + 1e-12
    return float(10 * np.log10(p_s / p_n))

def reduce_noise(y, sr=TARGET_SR, prop_decrease=0.7):
    # Step 6: spectral-gating noise reduction (gentle: 70% reduction, stationary noise).
    return nr.reduce_noise(y=y, sr=sr, stationary=True, prop_decrease=prop_decrease)

_meter = pyln.Meter(TARGET_SR)
def normalize_loudness(y, target_lufs=-23.0, sr=TARGET_SR):
    # Step 7: EBU R128 loudness normalisation, then peak-limit to avoid clipping.
    try:
        loud = _meter.integrated_loudness(y)
        if np.isfinite(loud):
            y = pyln.normalize.loudness(y, loud, target_lufs)
    except Exception:
        y = y / (np.sqrt(np.mean(y**2)) + 1e-9) * 0.05   # RMS fallback for very short clips
    peak = np.max(np.abs(y))
    return (y / peak * 0.99 if peak > 0.99 else y).astype(np.float32)

## 5. Quality metrics and pause (fluency) features

These are computed on the cleaned audio and saved as **features for the model**:

- `snr_db` — recording quality
- `clip_ratio` — fraction of clipped samples (distorted microphone input)
- `speech_ratio` — share of the clip that contains speech
- `n_pauses`, `mean_pause`, `max_pause`, `long_pauses` — hesitation patterns (pauses ≥ 0.3 s; long = ≥ 1 s)
- `n_speech_segments`, `mean_segment` — how fluently the speaker strings words together

In [ ]:
def clipping_ratio(y_raw, thresh=0.999):
    return float(np.mean(np.abs(y_raw) >= thresh))

def pause_features(y, sr=TARGET_SR, top_db=30, min_pause=0.3, long_pause=1.0):
    intervals = librosa.effects.split(y, top_db=top_db, frame_length=1024, hop_length=256)
    total = len(y) / sr
    if len(intervals) == 0:
        return dict(speech_ratio=0.0, n_pauses=0, mean_pause=0.0, max_pause=0.0,
                    long_pauses=0, n_speech_segments=0, mean_segment=0.0)
    seg_lens = (intervals[:, 1] - intervals[:, 0]) / sr
    gaps = (intervals[1:, 0] - intervals[:-1, 1]) / sr
    pauses = gaps[gaps >= min_pause]
    return dict(
        speech_ratio=float(seg_lens.sum() / total),
        n_pauses=int(len(pauses)),
        mean_pause=float(pauses.mean()) if len(pauses) else 0.0,
        max_pause=float(pauses.max()) if len(pauses) else 0.0,
        long_pauses=int((pauses >= long_pause).sum()),
        n_speech_segments=int(len(seg_lens)),
        mean_segment=float(seg_lens.mean()),
    )

## 6. Run the full preprocessing pipeline

`SNR_THRESHOLD`: noise reduction is applied **only to noisy clips** (SNR below the threshold). Denoising clean speech removes useful detail and can hurt transcription, so we avoid doing it blindly.

Cleaned files are saved as 16 kHz mono WAV in `/kaggle/working/clean_audio/`, and all features go into a CSV — so this cell only needs to run once.

In [ ]:
SNR_THRESHOLD = 15.0   # dB; clips noisier than this get gentle noise reduction

def preprocess_file(path, split):
    y_raw = load_audio(path)
    rec = {"clip_ratio": clipping_ratio(y_raw), "raw_duration": len(y_raw) / TARGET_SR}
    y = remove_dc_offset(y_raw)
    y = trim_edges(y)
    snr = estimate_snr(y)
    rec["snr_db"] = snr
    rec["denoised"] = snr < SNR_THRESHOLD
    if rec["denoised"]:
        y = reduce_noise(y)
    y = normalize_loudness(y)
    rec["clean_duration"] = len(y) / TARGET_SR
    rec.update(pause_features(y))
    out = os.path.join(CLEAN_DIR, split, os.path.splitext(os.path.basename(path))[0] + ".wav")
    os.makedirs(os.path.dirname(out), exist_ok=True)
    sf.write(out, y, TARGET_SR, subtype="PCM_16")
    rec["clean_path"] = out
    return rec

def run_pipeline(df, split):
    rows = []
    for p in tqdm(df["path"], desc=f"preprocessing {split}"):
        try:
            rows.append(preprocess_file(p, split) if isinstance(p, str) else {"clean_path": None})
        except Exception as e:
            print("FAILED:", p, e)
            rows.append({"clean_path": None})
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)

train_proc = run_pipeline(train_df, "train")
test_proc = run_pipeline(test_df, "test") if len(test_df) else pd.DataFrame()

train_proc.to_csv(f"{WORK_DIR}/train_audio_features.csv", index=False)
if len(test_proc): test_proc.to_csv(f"{WORK_DIR}/test_audio_features.csv", index=False)
print("Denoised clips -> train:", int(train_proc["denoised"].sum()),
      "| test:", int(test_proc["denoised"].sum()) if len(test_proc) else 0)

### Visual check: before vs after preprocessing

In [ ]:
import librosa.display
row = train_proc.sort_values("snr_db").iloc[0]          # the noisiest training clip
y_before = load_audio(row["path"])
y_after, _ = librosa.load(row["clean_path"], sr=TARGET_SR)

fig, axes = plt.subplots(2, 2, figsize=(16, 7))
for i, (y, title) in enumerate([(y_before, "Before"), (y_after, "After")]):
    librosa.display.waveshow(y, sr=TARGET_SR, ax=axes[0, i]); axes[0, i].set_title(f"{title} — waveform")
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(S, sr=TARGET_SR, x_axis="time", y_axis="hz", ax=axes[1, i])
    axes[1, i].set_title(f"{title} — spectrogram")
plt.suptitle(f"{os.path.basename(row['path'])}  (SNR {row['snr_db']:.1f} dB)"); plt.tight_layout(); plt.show()
display(Audio(y_before, rate=TARGET_SR)); display(Audio(y_after, rate=TARGET_SR))

In [ ]:
# Which audio features already relate to the grammar score? (Pearson correlation)
feat_cols = ["snr_db", "clip_ratio", "speech_ratio", "n_pauses", "mean_pause",
             "max_pause", "long_pauses", "n_speech_segments", "mean_segment", "clean_duration"]
if LABEL_COL:
    corr = train_proc[feat_cols + [LABEL_COL]].corr()[LABEL_COL].drop(LABEL_COL).sort_values()
    corr.plot(kind="barh", figsize=(8, 5), color="#4C72B0", title="Correlation with grammar score")
    plt.axvline(0, color="k", lw=.8); plt.show()

## 7. Verbatim transcription with faster-whisper

Whisper tends to **clean up** speech: it drops "um/uh", fixes repeated words, and sometimes even corrects grammar. That would hide exactly what we're scoring. Settings used to keep the transcript faithful:

- `temperature=0` — deterministic decoding, no creative rewriting
- `condition_on_previous_text=False` — prevents text from one window "leaking" into the next and reduces hallucinations
- `initial_prompt` with disfluencies — nudges Whisper to keep fillers (a known, documented trick)
- `vad_filter=False` — we already trimmed edges; keeping internal silence intact preserves timing
- `word_timestamps=True` — used for speech-rate and pause features

Model: `medium.en` on GPU (~20–30 min for all files). On CPU use `small.en` with `compute_type="int8"`.

In [ ]:
import torch
from faster_whisper import WhisperModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_SIZE = "medium.en" if DEVICE == "cuda" else "small.en"
asr = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type="float16" if DEVICE == "cuda" else "int8")
VERBATIM_PROMPT = "Umm, let me think like, hmm... Okay, so, uh, I I went to the the market and, you know, I buyed some."
print("Using", MODEL_SIZE, "on", DEVICE)

def transcribe(path):
    segments, info = asr.transcribe(
        path, language="en", beam_size=5, temperature=0.0,
        condition_on_previous_text=False, initial_prompt=VERBATIM_PROMPT,
        vad_filter=False, word_timestamps=True)
    words, texts, logprobs, nospeech = [], [], [], []
    for seg in segments:
        texts.append(seg.text.strip())
        logprobs.append(seg.avg_logprob); nospeech.append(seg.no_speech_prob)
        words.extend(seg.words or [])
    word_probs = [w.probability for w in words]
    return {
        "transcript": " ".join(texts).strip(),
        "asr_avg_logprob": float(np.mean(logprobs)) if logprobs else np.nan,
        "asr_no_speech": float(np.mean(nospeech)) if nospeech else np.nan,
        "asr_word_conf": float(np.mean(word_probs)) if word_probs else np.nan,
        "asr_low_conf_words": int(sum(p < 0.5 for p in word_probs)),
        "speech_start": words[0].start if words else 0.0,
        "speech_end": words[-1].end if words else 0.0,
    }

def transcribe_df(df, split):
    cache = f"{WORK_DIR}/{split}_transcripts.csv"
    if os.path.exists(cache):                     # resume safely if the session restarted
        print("Loaded cached", cache); return pd.read_csv(cache)
    out = [transcribe(p) if isinstance(p, str) else {"transcript": ""} for p in tqdm(df["clean_path"], desc=f"ASR {split}")]
    res = pd.concat([df.reset_index(drop=True), pd.DataFrame(out)], axis=1)
    res.to_csv(cache, index=False)
    return res

train_tx = transcribe_df(train_proc, "train")
test_tx = transcribe_df(test_proc, "test") if len(test_proc) else pd.DataFrame()
display(train_tx[[FILE_COL, LABEL_COL, "transcript"]].head() if LABEL_COL else train_tx[[FILE_COL, "transcript"]].head())

## 8. Transcript cleaning and text-based fluency features

We keep **two versions** of each transcript:
- `transcript` — verbatim (fillers, repetitions kept) → used for counting disfluencies
- `clean_text` — fillers removed, spacing/casing normalised → fed later to grammar checkers and language models, so they judge grammar rather than hesitation

Features:
- `n_words`, `speech_rate_wpm` (words per minute of whole clip), `articulation_rate` (words per minute of actual speaking time)
- `n_fillers`, `filler_rate` — um / uh / er / hmm / like / you know
- `n_repetitions` — immediately repeated words ("I I went")
- `type_token_ratio`, `mean_word_len` — vocabulary richness
- `n_sentences`, `mean_sentence_len` — sentence structure

In [ ]:
FILLERS = r"\b(um+|umm+|uh+|uhh+|er+|erm+|ah+|hmm+|mm+|like|you know|i mean|sort of|kind of)\b"

def clean_transcript(t):
    t = str(t) if isinstance(t, str) else ""
    t = re.sub(r"\b(um+|uh+|er+|erm+|ah+|hmm+|mm+)\b[,.]?", " ", t, flags=re.I)  # remove pure fillers only
    t = re.sub(r"\s+([,.?!])", r"\1", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t[:1].upper() + t[1:] if t else t

def text_features(row):
    t = str(row["transcript"]) if isinstance(row["transcript"], str) else ""
    tokens = re.findall(r"[a-zA-Z']+", t.lower())
    n = len(tokens)
    dur = max(row.get("clean_duration", 0) or 0, 1e-6)
    speak_time = max((row.get("clean_duration", 0) or 0) * (row.get("speech_ratio", 0) or 0), 1e-6)
    sents = [s for s in re.split(r"[.?!]+", t) if s.strip()]
    n_fill = len(re.findall(FILLERS, t.lower()))
    reps = sum(1 for a, b in zip(tokens, tokens[1:]) if a == b)
    return pd.Series({
        "n_words": n,
        "speech_rate_wpm": n / dur * 60,
        "articulation_rate": n / speak_time * 60,
        "n_fillers": n_fill,
        "filler_rate": n_fill / max(n, 1),
        "n_repetitions": reps,
        "type_token_ratio": len(set(tokens)) / max(n, 1),
        "mean_word_len": float(np.mean([len(w) for w in tokens])) if tokens else 0.0,
        "n_sentences": len(sents),
        "mean_sentence_len": n / max(len(sents), 1),
    })

for df in [train_tx, test_tx]:
    if not len(df): continue
    df["clean_text"] = df["transcript"].apply(clean_transcript)
    df[text_features(df.iloc[0]).index] = df.apply(text_features, axis=1)

# Flag likely ASR failures (empty or near-empty transcripts) for manual review
bad = train_tx[train_tx["n_words"] < 15]
print("Suspicious transcripts (<15 words):", len(bad))
display(bad[[FILE_COL, "transcript", "asr_no_speech"]].head())

## 9. Cross-validation folds for evaluation

With only 769 samples, a single train/validation split is unreliable. We create **5 stratified folds**: scores are binned so every fold has a similar mix of low, mid and high grammar scores. Every later model uses these same folds, so results are comparable.

Evaluation metrics to report per fold (mean ± std): **Pearson r** (primary), **Spearman ρ**, **RMSE**, **MAE**.

In [ ]:
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5
train_tx["fold"] = -1
if LABEL_COL:
    bins = pd.cut(train_tx[LABEL_COL], bins=[-np.inf, 1.5, 2.5, 3.5, 4.5, np.inf], labels=False)
    # merge rare bins so stratification never fails
    counts = bins.value_counts()
    bins = bins.replace({b: counts.idxmax() for b in counts[counts < N_FOLDS].index})
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for f, (_, va) in enumerate(skf.split(train_tx, bins)):
        train_tx.loc[va, "fold"] = f
    print(train_tx.groupby("fold")[LABEL_COL].agg(["count", "mean", "std"]).round(3))

In [ ]:
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate(y_true, y_pred):
    # Standard metric set for this task. Predictions are clipped to the valid 0-5 range.
    y_pred = np.clip(y_pred, 0, 5)
    return {
        "pearson": pearsonr(y_true, y_pred)[0],
        "spearman": spearmanr(y_true, y_pred)[0],
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }

# Sanity baseline: predicting the training mean (Pearson undefined, RMSE = label std)
if LABEL_COL:
    y = train_tx[LABEL_COL].values
    print("Mean-prediction baseline RMSE:", round(float(np.sqrt(np.mean((y - y.mean()) ** 2))), 3))

## 10. Save everything for the modelling notebook

In [ ]:
train_tx.to_csv(f"{WORK_DIR}/train_preprocessed.csv", index=False)
if len(test_tx): test_tx.to_csv(f"{WORK_DIR}/test_preprocessed.csv", index=False)

summary = {
    "train_clips": int(len(train_tx)), "test_clips": int(len(test_tx)),
    "sample_rate": TARGET_SR, "snr_threshold_db": SNR_THRESHOLD,
    "denoised_train": int(train_tx["denoised"].sum()),
    "asr_model": MODEL_SIZE, "folds": N_FOLDS,
}
print(json.dumps(summary, indent=2))
print("\nSaved:", [f for f in os.listdir(WORK_DIR) if f.endswith(".csv")])

## Summary — what this notebook produces

| Output | Used for |
|---|---|
| `clean_audio/` (16 kHz mono WAV) | Audio models (wav2vec2 / WavLM embeddings) |
| `train/test_preprocessed.csv` | Transcripts + clean text + all features + CV folds |
| Audio features | SNR, clipping, speech ratio, pauses |
| ASR features | Whisper confidence (low confidence often = unclear or ungrammatical speech) |
| Text features | Speech rate, fillers, repetitions, vocabulary, sentence length |

**Next step (modelling notebook):** grammar-error features from LanguageTool on `clean_text`, sentence embeddings (e.g. DeBERTa / sentence-transformers), then Ridge / SVR / LightGBM with the 5 folds above, evaluated with Pearson, Spearman, RMSE and MAE.

**Limitations to mention in your report:** Whisper may still silently correct some grammar errors; SNR is a rough estimate; filler detection with words like "like" can over-count.